# 🧬 ConsortiumTuner v3 — 6 modules, 20 circuits, live simulation

**BE552 Project**

---

## What's new in v3

- **Exactly 20 canonical circuits** — names match `whole_ecosystem_20circuits.xml` (the SBML ecosystem model in iBioSim)
- **6 modules** (not 7) — nitrification and denitrification collapsed into a single **Nitrogen cycle** module, matching the SBML
- **Genetic parts reference** — a read-only table showing the actual GOLDBAR parts (promoter / RBS / CDS / terminator) for each of the 20 circuits, grouped by module
- **One big iBioSim-style plot** with a species picker (presets + 33 individual checkboxes, just like iBioSim's *Edit Graph → run-1* dialog)
- **Live re-integration** — the moment you move a slider, the ecosystem re-integrates and the trajectories redraw

## The 6 modules

| # | Module | Circuits in library |
|---|---|---|
| 1 | **Cross-feeding** | Ecoli_K · Ecoli_L · Ecoli_Trp · Ecoli_His |
| 2 | **Nitrogen cycle** | Nitrosomonas · Nitrobacter · Pseudomonas_stutzeri · Paracoccus_denitrificans |
| 3 | **Electrogenesis** | StrainA_Gsulfurreducens · StrainB_Gmetalireducens · OmcZ_Booster_StrainA · OmcZ_Booster_StrainB |
| 4 | **Quorum sensing** | Luxl_LuxR · LuxS |
| 5 | **Kill switch** | KillSwitch_StrainA · KillSwitch_StrainB |
| 6 | **Metabolite production** | BetaCarotene · Isoprenoid · Naringenin · MalonylCoA |

Each slider scales the **rate constants** of its module between 0 (module off) and 1 (module at native strength). The selected consortium + live dynamics update together.


## Step 1 — Imports

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

plt.rcParams['figure.dpi'] = 100
print("Imports OK")

Imports OK


## Step 2 — Circuit library (20 circuits, 6 modules)

Every circuit scores `1.0` on its module axis and `0.0` elsewhere. Dependencies
(e.g. `KillSwitch_StrainA` requires `StrainA_Gsulfurreducens`) are enforced
in the selection step.


In [ ]:
MODULES = [
    'cross_feeding', 'nitrogen_cycle', 'electrogenesis',
    'quorum_sensing', 'kill_switch', 'metabolite_production'
]

def _scores(active):
    """Helper: 1.0 on the active module, 0.0 elsewhere."""
    return {m: (1.0 if m == active else 0.0) for m in MODULES}

CIRCUITS = {
    # ===== 1. CROSS-FEEDING (4) =====
    'Ecoli_K': {
        'organism': 'Escherichia coli (lys+ / leu-)',
        'module': 'cross_feeding',
        'spec': 'Ptrc then RBS2 then lysC then argD then lysA then T2',
        'note': 'Lysine producer, Leu auxotroph',
        'scores': _scores('cross_feeding'), 'requires': []
    },
    'Ecoli_L': {
        'organism': 'Escherichia coli (leu+ / lys-)',
        'module': 'cross_feeding',
        'spec': 'Ptrc then RBS1 then leuB then leuC then leuD then T1',
        'note': 'Leucine producer, Lys auxotroph',
        'scores': _scores('cross_feeding'), 'requires': []
    },
    'Ecoli_Trp': {
        'organism': 'Escherichia coli (trp+ / his-)',
        'module': 'cross_feeding',
        'spec': 'Ptrc then RBS1 then trpB then trpA then T1',
        'note': 'Tryptophan producer, His auxotroph',
        'scores': _scores('cross_feeding'), 'requires': []
    },
    'Ecoli_His': {
        'organism': 'Escherichia coli (his+ / trp-)',
        'module': 'cross_feeding',
        'spec': 'Ptrc then RBS1 then hisC then hisD then T1',
        'note': 'Histidine producer, Trp auxotroph',
        'scores': _scores('cross_feeding'), 'requires': []
    },

    # ===== 2. NITROGEN CYCLE (4) =====
    'Nitrosomonas': {
        'organism': 'Nitrosomonas europaea',
        'module': 'nitrogen_cycle',
        'spec': 'Ptrc then RBS1 then amoA then hao then T1',
        'note': 'NH4+ -> NO2- (amoA, hao)',
        'scores': _scores('nitrogen_cycle'), 'requires': []
    },
    'Nitrobacter': {
        'organism': 'Nitrobacter winogradskyi',
        'module': 'nitrogen_cycle',
        'spec': 'Ptrc then RBS2 then nxrA then T2',
        'note': 'NO2- -> NO3- (nxrA)',
        'scores': _scores('nitrogen_cycle'), 'requires': ['Nitrosomonas']
    },
    'Pseudomonas_stutzeri': {
        'organism': 'Pseudomonas stutzeri',
        'module': 'nitrogen_cycle',
        'spec': 'Ptrc then RBS1 then narG then nirS then T1',
        'note': 'NO3- -> NO2- -> NO (early denitrification)',
        'scores': _scores('nitrogen_cycle'), 'requires': []
    },
    'Paracoccus_denitrificans': {
        'organism': 'Paracoccus denitrificans',
        'module': 'nitrogen_cycle',
        'spec': 'Ptrc then RBS2 then norB then nosZ then T2',
        'note': 'NO -> N2O -> N2 (late denitrification)',
        'scores': _scores('nitrogen_cycle'), 'requires': ['Pseudomonas_stutzeri']
    },

    # ===== 3. ELECTROGENESIS (4) =====
    'StrainA_Gsulfurreducens': {
        'organism': 'Geobacter sulfurreducens',
        'module': 'electrogenesis',
        'spec': 'Ptrc then RBS1 then acs then Term1',
        'note': 'acs expression; anode-reducing chassis',
        'scores': _scores('electrogenesis'), 'requires': []
    },
    'StrainB_Gmetalireducens': {
        'organism': 'Geobacter metallireducens',
        'module': 'electrogenesis',
        'spec': 'PnirH then RBS2 then omcZ then Term2',
        'note': 'PnirH-driven omcZ; NO-responsive EET',
        'scores': _scores('electrogenesis'), 'requires': []
    },
    'OmcZ_Booster_StrainA': {
        'organism': 'Geobacter sulfurreducens',
        'module': 'electrogenesis',
        'spec': 'PomcZ then RBS1 then omcZ then Term1',
        'note': 'Extra PomcZ->omcZ cassette in Strain A',
        'scores': _scores('electrogenesis'), 'requires': ['StrainA_Gsulfurreducens']
    },
    'OmcZ_Booster_StrainB': {
        'organism': 'Geobacter metallireducens',
        'module': 'electrogenesis',
        'spec': 'PomcZ then RBS2 then omcZ then Term2',
        'note': 'Extra PomcZ->omcZ cassette in Strain B',
        'scores': _scores('electrogenesis'), 'requires': ['StrainB_Gmetalireducens']
    },

    # ===== 4. QUORUM SENSING (2) =====
    'Luxl_LuxR': {
        'organism': 'LuxI/LuxR-carrying chassis',
        'module': 'quorum_sensing',
        'spec': 'Plux then RBS1 then luxI then luxR then T1',
        'note': 'AHL production + sensing (drives KillSignal)',
        'scores': _scores('quorum_sensing'), 'requires': []
    },
    'LuxS': {
        'organism': 'LuxS-carrying chassis',
        'module': 'quorum_sensing',
        'spec': 'Ptrc then RBS1 then luxS then T1',
        'note': 'AI-2 universal interspecies QS signal',
        'scores': _scores('quorum_sensing'), 'requires': []
    },

    # ===== 5. KILL SWITCH (2) =====
    'KillSwitch_StrainA': {
        'organism': 'Geobacter sulfurreducens',
        'module': 'kill_switch',
        'spec': '(Pconst then RBS1 then mazE then Term1) then (Prep then RBS1 then mazF then Term1)',
        'note': 'MazE/MazF toxin-antitoxin in Strain A',
        'scores': _scores('kill_switch'), 'requires': ['StrainA_Gsulfurreducens']
    },
    'KillSwitch_StrainB': {
        'organism': 'Geobacter metallireducens',
        'module': 'kill_switch',
        'spec': '(Pconst then RBS2 then mazE then Term2) then (Prep then RBS2 then mazF then Term2)',
        'note': 'Orthogonal MazE/MazF in Strain B',
        'scores': _scores('kill_switch'), 'requires': ['StrainB_Gmetalireducens']
    },

    # ===== 6. METABOLITE PRODUCTION (4) =====
    'BetaCarotene': {
        'organism': 'Engineered carotenoid producer',
        'module': 'metabolite_production',
        'spec': 'Ptrc then RBS1 then crtE then crtB then crtI then T1',
        'note': 'Isoprenoid -> BetaCarotene',
        'scores': _scores('metabolite_production'), 'requires': []
    },
    'Isoprenoid': {
        'organism': 'Engineered isoprenoid producer',
        'module': 'metabolite_production',
        'spec': 'Ptrc then RBS1 then atoB then HMGS then HMGR then T1',
        'note': 'AcetylCoA -> Isoprenoid (mevalonate pathway)',
        'scores': _scores('metabolite_production'), 'requires': []
    },
    'Naringenin': {
        'organism': 'Engineered flavonoid producer',
        'module': 'metabolite_production',
        'spec': 'Ptrc then RBS1 then TAL then 4CL then CHS then CHI then T1',
        'note': 'MalonylCoA -> Naringenin',
        'scores': _scores('metabolite_production'), 'requires': []
    },
    'MalonylCoA': {
        'organism': 'Engineered malonyl-CoA producer',
        'module': 'metabolite_production',
        'spec': 'PfapR then RBS1 then fapR then fabH then T1',
        'note': 'AcetylCoA -> MalonylCoA (fapR/fabH)',
        'scores': _scores('metabolite_production'), 'requires': []
    },
}

# Sanity
assert len(CIRCUITS) == 20, f"Expected 20 circuits, got {len(CIRCUITS)}"
print(f"Loaded {len(CIRCUITS)} circuits across {len(MODULES)} modules:")
for m in MODULES:
    members = [n for n,c in CIRCUITS.items() if c['module'] == m]
    print(f"  {m:25s} ({len(members)}): {', '.join(members)}")

Loaded 20 circuits across 6 modules:
  cross_feeding             (4): Ecoli_K, Ecoli_L, Ecoli_Trp, Ecoli_His
  nitrogen_cycle            (4): Nitrosomonas, Nitrobacter, Pseudomonas_stutzeri, Paracoccus_denitrificans
  electrogenesis            (4): StrainA_Gsulfurreducens, StrainB_Gmetalireducens, OmcZ_Booster_StrainA, OmcZ_Booster_StrainB
  quorum_sensing            (2): Luxl_LuxR, LuxS
  kill_switch               (2): KillSwitch_StrainA, KillSwitch_StrainB
  metabolite_production     (4): BetaCarotene, Isoprenoid, Naringenin, MalonylCoA


## Step 3 — Genetic parts reference

Each circuit is a **transcriptional unit**: `Promoter → RBS → CDS₁ → … → CDSₙ → Terminator`.
Kill-switch circuits are composite (two transcriptional units in tandem).

The table below shows the **actual genetic parts** used to build each of the 20 circuits,
grouped by module. This is a reference view — tuning happens with the module sliders
in Step 7.


In [ ]:
# ---- Library of interchangeable parts ----
AVAILABLE_PARTS = {
    'promoter':   ['Ptrc', 'Plux', 'PfapR', 'PnirH', 'PomcZ', 'Pconst', 'Prep', 'PlacI', 'PT7'],
    'rbs':        ['RBS1', 'RBS2', 'RBSstrong', 'RBSweak'],
    'terminator': ['T1', 'T2', 'Term1', 'Term2', 'TrrnB', 'TL3'],
}

# ---- Structured parts per circuit (lists support composite circuits) ----
CIRCUIT_PARTS = {
    # Cross-feeding
    'Ecoli_K':   [{'promoter':'Ptrc','rbs':'RBS2','cds':['lysC','argD','lysA'],'terminator':'T2'}],
    'Ecoli_L':   [{'promoter':'Ptrc','rbs':'RBS1','cds':['leuB','leuC','leuD'],'terminator':'T1'}],
    'Ecoli_Trp': [{'promoter':'Ptrc','rbs':'RBS1','cds':['trpB','trpA'],       'terminator':'T1'}],
    'Ecoli_His': [{'promoter':'Ptrc','rbs':'RBS1','cds':['hisC','hisD'],       'terminator':'T1'}],
    # Nitrogen cycle
    'Nitrosomonas':             [{'promoter':'Ptrc','rbs':'RBS1','cds':['amoA','hao'], 'terminator':'T1'}],
    'Nitrobacter':              [{'promoter':'Ptrc','rbs':'RBS2','cds':['nxrA'],       'terminator':'T2'}],
    'Pseudomonas_stutzeri':     [{'promoter':'Ptrc','rbs':'RBS1','cds':['narG','nirS'],'terminator':'T1'}],
    'Paracoccus_denitrificans': [{'promoter':'Ptrc','rbs':'RBS2','cds':['norB','nosZ'],'terminator':'T2'}],
    # Electrogenesis
    'StrainA_Gsulfurreducens':  [{'promoter':'Ptrc', 'rbs':'RBS1','cds':['acs'],  'terminator':'Term1'}],
    'StrainB_Gmetalireducens':  [{'promoter':'PnirH','rbs':'RBS2','cds':['omcZ'], 'terminator':'Term2'}],
    'OmcZ_Booster_StrainA':     [{'promoter':'PomcZ','rbs':'RBS1','cds':['omcZ'], 'terminator':'Term1'}],
    'OmcZ_Booster_StrainB':     [{'promoter':'PomcZ','rbs':'RBS2','cds':['omcZ'], 'terminator':'Term2'}],
    # Quorum sensing
    'Luxl_LuxR': [{'promoter':'Plux','rbs':'RBS1','cds':['luxI','luxR'],'terminator':'T1'}],
    'LuxS':      [{'promoter':'Ptrc','rbs':'RBS1','cds':['luxS'],       'terminator':'T1'}],
    # Kill switches (composite: antitoxin + toxin cassettes)
    'KillSwitch_StrainA': [
        {'promoter':'Pconst','rbs':'RBS1','cds':['mazE'],'terminator':'Term1'},
        {'promoter':'Prep',  'rbs':'RBS1','cds':['mazF'],'terminator':'Term1'},
    ],
    'KillSwitch_StrainB': [
        {'promoter':'Pconst','rbs':'RBS2','cds':['mazE'],'terminator':'Term2'},
        {'promoter':'Prep',  'rbs':'RBS2','cds':['mazF'],'terminator':'Term2'},
    ],
    # Metabolite production
    'BetaCarotene': [{'promoter':'Ptrc', 'rbs':'RBS1','cds':['crtE','crtB','crtI'],    'terminator':'T1'}],
    'Isoprenoid':   [{'promoter':'Ptrc', 'rbs':'RBS1','cds':['atoB','HMGS','HMGR'],    'terminator':'T1'}],
    'Naringenin':   [{'promoter':'Ptrc', 'rbs':'RBS1','cds':['TAL','4CL','CHS','CHI'], 'terminator':'T1'}],
    'MalonylCoA':   [{'promoter':'PfapR','rbs':'RBS1','cds':['fapR','fabH'],           'terminator':'T1'}],
}

assert set(CIRCUIT_PARTS) == set(CIRCUITS), "Parts must be defined for every circuit"


def unit_to_spec(u):
    return ' then '.join([u['promoter'], u['rbs'], *u['cds'], u['terminator']])

def circuit_to_spec(cname):
    specs = [unit_to_spec(u) for u in CIRCUIT_PARTS[cname]]
    return specs[0] if len(specs) == 1 else ' then '.join(f'({s})' for s in specs)

# Sync the CIRCUITS dict's spec field with CIRCUIT_PARTS on load
for cname in CIRCUITS:
    CIRCUITS[cname]['spec'] = circuit_to_spec(cname)

print('Parts defined for', len(CIRCUIT_PARTS), 'circuits.')
print('Sample:', 'Ecoli_K ->', circuit_to_spec('Ecoli_K'))
print('Sample:', 'KillSwitch_StrainA ->', circuit_to_spec('KillSwitch_StrainA'))

Parts defined for 20 circuits.
Sample: Ecoli_K -> Ptrc then RBS2 then lysC then argD then lysA then T2
Sample: KillSwitch_StrainA -> (Pconst then RBS1 then mazE then Term1) then (Prep then RBS1 then mazF then Term1)


In [ ]:
# ---- Interactive parts editor widget ----
MODULE_ORDER = ['cross_feeding','nitrogen_cycle','electrogenesis',
                'quorum_sensing','kill_switch','metabolite_production']
MODULE_LABEL = {
    'cross_feeding':         '1️⃣  Cross-feeding',
    'nitrogen_cycle':        '2️⃣  Nitrogen cycle',
    'electrogenesis':        '3️⃣  Electrogenesis',
    'quorum_sensing':        '4️⃣  Quorum sensing',
    'kill_switch':           '5️⃣  Kill switch',
    'metabolite_production': '6️⃣  Metabolite production',
}

# --- Render as static read-only rows (no dropdowns — reference only) ---
def _static_row(cname, unit_idx, is_first):
    u = CIRCUIT_PARTS[cname][unit_idx]
    name_html = (f'<b>{cname}</b>' if is_first
                 else '<i style="color:#888">  &nbsp;↳ unit 2</i>')
    cell_style = ('background:#e2e8f0;color:#0f172a;padding:5px 10px;'
                  'border-radius:4px;font-family:monospace;')
    return widgets.HTML(
        f'<div style="display:flex;gap:6px;align-items:center;padding:2px 0;">'
        f'  <div style="width:220px">{name_html}</div>'
        f'  <div style="{cell_style}width:90px;">{u["promoter"]}</div>'
        f'  <div style="{cell_style}width:90px;">{u["rbs"]}</div>'
        f'  <div style="{cell_style}flex:1;min-width:240px;">{", ".join(u["cds"])}</div>'
        f'  <div style="{cell_style}width:90px;">{u["terminator"]}</div>'
        f'</div>'
    )


def build_parts_editor():
    out = []
    # Column headers
    hdr_style = 'color:#334155;font-weight:bold;'
    out.append(widgets.HTML(
        f'<div style="display:flex;gap:6px;padding:4px 0;border-bottom:1px solid #cbd5e1;">'
        f'  <div style="width:220px;{hdr_style}">Circuit</div>'
        f'  <div style="width:90px;{hdr_style}">Promoter</div>'
        f'  <div style="width:90px;{hdr_style}">RBS</div>'
        f'  <div style="flex:1;min-width:240px;{hdr_style}">CDS list</div>'
        f'  <div style="width:90px;{hdr_style}">Terminator</div>'
        f'</div>'
    ))

    for mod in MODULE_ORDER:
        circuits_in_mod = [n for n, c in CIRCUITS.items() if c['module'] == mod]
        if not circuits_in_mod: continue
        out.append(widgets.HTML(
            f'<div style="background:#1e293b;color:white;padding:6px 10px;'
            f'margin-top:10px;border-radius:4px;"><b>{MODULE_LABEL[mod]}</b> '
            f'<span style="color:#94a3b8;">({len(circuits_in_mod)} circuits)</span></div>'
        ))
        for cname in circuits_in_mod:
            for i in range(len(CIRCUIT_PARTS[cname])):
                out.append(_static_row(cname, i, is_first=(i == 0)))
            # GOLDBAR spec line under each circuit
            out.append(widgets.HTML(
                f'<div style="padding-left:220px;color:#64748b;font-size:11px;'
                f'margin-bottom:6px;">→ GOLDBAR: '
                f'<code style="color:#0ea5e9;background:#f1f5f9;'
                f'padding:3px 6px;border-radius:4px;">{CIRCUITS[cname]["spec"]}</code></div>'
            ))

    return widgets.VBox(out)

parts_editor = build_parts_editor()
display(widgets.HTML(
    '<h3 style="margin:0 0 6px 0;">🧬 Genetic parts table '
    '(20 circuits · 22 transcriptional units)</h3>'
    '<p style="color:#64748b;margin:0 0 10px 0;">'
    'Reference only — this shows the actual GOLDBAR parts for each circuit. '
    'Tuning happens with the module sliders further down.</p>'
))
display(parts_editor)

HTML(value='<h3 style="margin:0 0 6px 0;">🧬 Genetic parts table (20 circuits · 22 transcriptional units)</h3><…

## Step 4 — The ODE model

Mirror of the SBML `whole_ecosystem_20circuits.xml` model: 33 species,
49 reactions. Each slider multiplies the rate constants of its module by a
value in `[0, 1]`, so moving a slider to `0` effectively switches that
module off, and `1` gives the native dynamics.


In [ ]:
# ----- Species (33 state variables) -----
SPECIES = [
    # Cross-feeding
    'Ecoli_K','Ecoli_L','Ecoli_Trp','Ecoli_His',
    'Lys','Leu','Trp','His',
    # Electrogenesis
    'Gsulf','Gmet','Acs','OmcZ_A','OmcZ_B','Current',
    # Kill switch
    'MazE_A','MazF_A','MazE_B','MazF_B','KillSignal',
    # Quorum sensing
    'AHL','LuxR_AHL','AI2',
    # Nitrogen cycle
    'NH4','NO2','NO3','NO','N2O','N2',
    # Metabolite production
    'AcetylCoA','MalonylCoA','Naringenin','Isoprenoid','BetaCarotene',
]
IDX = {s:i for i,s in enumerate(SPECIES)}

# Initial conditions
Y0 = np.zeros(len(SPECIES))
Y0[IDX['Ecoli_K']]   = 1.0
Y0[IDX['Ecoli_L']]   = 1.0
Y0[IDX['Ecoli_Trp']] = 1.0
Y0[IDX['Ecoli_His']] = 1.0
Y0[IDX['Gsulf']]     = 1.0
Y0[IDX['Gmet']]      = 1.0
Y0[IDX['MazE_A']]    = 5.0
Y0[IDX['MazE_B']]    = 5.0
Y0[IDX['NH4']]       = 10.0
Y0[IDX['AcetylCoA']] = 5.0

# ----- Base rate constants (same values as the SBML model) -----
# Each rate is tagged with the module whose slider scales it.
RATES = {
    # cross-feeding
    'k_K_lys':    (0.15, 'cross_feeding'),
    'k_L_leu':    (0.15, 'cross_feeding'),
    'k_Trp_trp':  (0.15, 'cross_feeding'),
    'k_His_his':  (0.15, 'cross_feeding'),
    'k_K_grow':   (0.04, 'cross_feeding'),
    'k_L_grow':   (0.04, 'cross_feeding'),
    'k_Trp_grow': (0.04, 'cross_feeding'),
    'k_His_grow': (0.04, 'cross_feeding'),
    'k_K_death':   (0.05, 'cross_feeding'),
    'k_L_death':   (0.05, 'cross_feeding'),
    'k_Trp_death': (0.05, 'cross_feeding'),
    'k_His_death': (0.05, 'cross_feeding'),
    # electrogenesis
    'k_Gsulf_grow':    (0.03, 'electrogenesis'),
    'k_Gmet_grow':     (0.03, 'electrogenesis'),
    'k_Gsulf_death':   (0.10, 'electrogenesis'),
    'k_Gmet_death':    (0.10, 'electrogenesis'),
    'k_acs':           (0.10, 'electrogenesis'),
    'k_omcZ_B_basal':  (0.05, 'electrogenesis'),
    'k_omcZ_boost_A':  (0.20, 'electrogenesis'),
    'k_omcZ_boost_B':  (0.20, 'electrogenesis'),
    'k_current':       (0.02, 'electrogenesis'),
    'k_NO_induces_omcZ': (0.08, 'electrogenesis'),
    # kill switch
    'k_mazE_A':   (0.10, 'kill_switch'),
    'k_mazE_B':   (0.10, 'kill_switch'),
    'k_mazF_A':   (0.15, 'kill_switch'),
    'k_mazF_B':   (0.15, 'kill_switch'),
    'k_antitox':  (0.50, 'kill_switch'),
    'k_tox_kill': (0.25, 'kill_switch'),
    'k_mazE_deg': (0.05, 'kill_switch'),
    # quorum sensing
    'k_luxI':        (0.05, 'quorum_sensing'),
    'k_luxS':        (0.05, 'quorum_sensing'),
    'k_luxR_act':    (0.10, 'quorum_sensing'),
    'k_killsig':     (0.05, 'quorum_sensing'),
    'k_AHL_deg':     (0.02, 'quorum_sensing'),
    'k_AI2_deg':     (0.02, 'quorum_sensing'),
    'k_killsig_deg': (0.05, 'quorum_sensing'),
    # nitrogen cycle
    'k_nitrosomonas': (0.15, 'nitrogen_cycle'),
    'k_nitrobacter':  (0.12, 'nitrogen_cycle'),
    'k_pseud_narG':   (0.10, 'nitrogen_cycle'),
    'k_pseud_nirS':   (0.10, 'nitrogen_cycle'),
    'k_parac_norB':   (0.10, 'nitrogen_cycle'),
    'k_parac_nosZ':   (0.10, 'nitrogen_cycle'),
    # metabolite production
    'k_malCoA':     (0.08, 'metabolite_production'),
    'k_naringenin': (0.06, 'metabolite_production'),
    'k_isoprenoid': (0.08, 'metabolite_production'),
    'k_betacarot':  (0.06, 'metabolite_production'),
}

def effective_rates(weights):
    """Scale each rate constant by its module's slider weight."""
    return {k: base * weights[mod] for k,(base, mod) in RATES.items()}

def rhs(t, y, p):
    """33-equation ODE system. `p` = effective_rates(weights)."""
    # Unpack (keeps formulas readable)
    Ecoli_K, Ecoli_L, Ecoli_Trp, Ecoli_His = y[0], y[1], y[2], y[3]
    Lys, Leu, Trp, His = y[4], y[5], y[6], y[7]
    Gsulf, Gmet, Acs, OmcZ_A, OmcZ_B, Current = y[8], y[9], y[10], y[11], y[12], y[13]
    MazE_A, MazF_A, MazE_B, MazF_B, KillSignal = y[14], y[15], y[16], y[17], y[18]
    AHL, LuxR_AHL, AI2 = y[19], y[20], y[21]
    NH4, NO2, NO3, NO_, N2O, N2 = y[22], y[23], y[24], y[25], y[26], y[27]
    AcetylCoA, MalonylCoA, Naringenin, Isoprenoid, BetaCarotene = y[28], y[29], y[30], y[31], y[32]

    # Clamp to non-negative for rate calculations
    Ecoli_K = max(Ecoli_K, 0); Ecoli_L = max(Ecoli_L, 0)
    Ecoli_Trp = max(Ecoli_Trp, 0); Ecoli_His = max(Ecoli_His, 0)
    Gsulf = max(Gsulf, 0); Gmet = max(Gmet, 0)

    # --- Cross-feeding reactions ---
    v_K_lys   = p['k_K_lys']   * Ecoli_K
    v_L_leu   = p['k_L_leu']   * Ecoli_L
    v_Trp_prod= p['k_Trp_trp'] * Ecoli_Trp
    v_His_prod= p['k_His_his'] * Ecoli_His
    v_K_grow   = p['k_K_grow']   * Ecoli_K   * max(Leu, 0)
    v_L_grow   = p['k_L_grow']   * Ecoli_L   * max(Lys, 0)
    v_Trp_grow = p['k_Trp_grow'] * Ecoli_Trp * max(His, 0)
    v_His_grow = p['k_His_grow'] * Ecoli_His * max(Trp, 0)
    v_K_death   = p['k_K_death']   * Ecoli_K
    v_L_death   = p['k_L_death']   * Ecoli_L
    v_Trp_death = p['k_Trp_death'] * Ecoli_Trp
    v_His_death = p['k_His_death'] * Ecoli_His

    # --- Electrogenesis ---
    v_Gsulf_grow  = p['k_Gsulf_grow']  * Gsulf
    v_Gmet_grow   = p['k_Gmet_grow']   * Gmet
    v_Gsulf_death = p['k_Gsulf_death'] * Gsulf
    v_Gmet_death  = p['k_Gmet_death']  * Gmet
    v_acs         = p['k_acs']         * Gsulf
    v_omcZ_B_base = p['k_omcZ_B_basal']* Gmet
    v_omcZ_B_ind  = p['k_NO_induces_omcZ'] * Gmet * max(NO_, 0)
    v_omcZ_bst_A  = p['k_omcZ_boost_A']* Gsulf
    v_omcZ_bst_B  = p['k_omcZ_boost_B']* Gmet
    v_current     = p['k_current'] * Gsulf * Gmet * (1 + max(OmcZ_A,0)) * (1 + max(OmcZ_B,0))

    # --- Kill switch ---
    v_mazE_A   = p['k_mazE_A']   * Gsulf
    v_mazF_A   = p['k_mazF_A']   * max(KillSignal,0) * Gsulf
    v_antitox_A= p['k_antitox']  * max(MazE_A,0) * max(MazF_A,0)
    v_kill_A   = p['k_tox_kill'] * max(MazF_A,0) * Gsulf
    v_mazE_A_dg= p['k_mazE_deg'] * max(MazE_A,0)
    v_mazE_B   = p['k_mazE_B']   * Gmet
    v_mazF_B   = p['k_mazF_B']   * max(KillSignal,0) * Gmet
    v_antitox_B= p['k_antitox']  * max(MazE_B,0) * max(MazF_B,0)
    v_kill_B   = p['k_tox_kill'] * max(MazF_B,0) * Gmet
    v_mazE_B_dg= p['k_mazE_deg'] * max(MazE_B,0)

    # --- Quorum sensing ---
    v_AHL     = p['k_luxI']     * Gsulf
    v_luxR    = p['k_luxR_act'] * max(AHL,0)
    v_killsig = p['k_killsig']  * max(LuxR_AHL,0)
    v_AHL_dg  = p['k_AHL_deg']  * max(AHL,0)
    v_ksig_dg = p['k_killsig_deg'] * max(KillSignal,0)
    v_AI2     = p['k_luxS']     * Gmet
    v_AI2_dg  = p['k_AI2_deg']  * max(AI2,0)

    # --- Nitrogen cycle ---
    v_nitroso = p['k_nitrosomonas'] * max(NH4,0)
    v_nitroba = p['k_nitrobacter']  * max(NO2,0)
    v_narG    = p['k_pseud_narG']   * max(NO3,0)
    v_nirS    = p['k_pseud_nirS']   * max(NO2,0)
    v_norB    = p['k_parac_norB']   * max(NO_,0)
    v_nosZ    = p['k_parac_nosZ']   * max(N2O,0)

    # --- Metabolite production ---
    v_malCoA    = p['k_malCoA']     * max(AcetylCoA,0)
    v_naringen  = p['k_naringenin'] * max(MalonylCoA,0)
    v_isoprenoid= p['k_isoprenoid'] * max(AcetylCoA,0)
    v_betacarot = p['k_betacarot']  * max(Isoprenoid,0)

    # --- Derivatives ---
    dy = np.zeros(len(SPECIES))
    # Cross-feeding: production doesn't consume the strain (catalytic),
    # growth consumes substrate AA and doubles the strain, death removes.
    dy[IDX['Ecoli_K']]   = v_K_grow   - v_K_death
    dy[IDX['Ecoli_L']]   = v_L_grow   - v_L_death
    dy[IDX['Ecoli_Trp']] = v_Trp_grow - v_Trp_death
    dy[IDX['Ecoli_His']] = v_His_grow - v_His_death
    dy[IDX['Lys']] = v_K_lys    - v_L_grow
    dy[IDX['Leu']] = v_L_leu    - v_K_grow
    dy[IDX['Trp']] = v_Trp_prod - v_His_grow
    dy[IDX['His']] = v_His_prod - v_Trp_grow

    # Electrogenesis
    dy[IDX['Gsulf']]   = v_Gsulf_grow - v_Gsulf_death - v_kill_A
    dy[IDX['Gmet']]    = v_Gmet_grow  - v_Gmet_death  - v_kill_B
    dy[IDX['Acs']]     = v_acs
    dy[IDX['OmcZ_A']]  = v_omcZ_bst_A
    dy[IDX['OmcZ_B']]  = v_omcZ_B_base + v_omcZ_B_ind + v_omcZ_bst_B
    dy[IDX['Current']] = v_current

    # Kill switch
    dy[IDX['MazE_A']] = v_mazE_A - v_antitox_A - v_mazE_A_dg
    dy[IDX['MazF_A']] = v_mazF_A - v_antitox_A
    dy[IDX['MazE_B']] = v_mazE_B - v_antitox_B - v_mazE_B_dg
    dy[IDX['MazF_B']] = v_mazF_B - v_antitox_B
    dy[IDX['KillSignal']] = v_killsig - v_ksig_dg

    # Quorum sensing
    dy[IDX['AHL']]      = v_AHL - v_luxR - v_AHL_dg
    dy[IDX['LuxR_AHL']] = v_luxR
    dy[IDX['AI2']]      = v_AI2 - v_AI2_dg

    # Nitrogen cycle (NH4 -> NO2 -> NO3 -> NO2 -> NO -> N2O -> N2)
    dy[IDX['NH4']] = -v_nitroso
    dy[IDX['NO2']] =  v_nitroso - v_nitroba + v_narG - v_nirS
    dy[IDX['NO3']] =  v_nitroba - v_narG
    dy[IDX['NO']]  =  v_nirS    - v_norB
    dy[IDX['N2O']] =  v_norB    - v_nosZ
    dy[IDX['N2']]  =  v_nosZ

    # Metabolite production
    dy[IDX['AcetylCoA']]    = -v_malCoA - v_isoprenoid
    dy[IDX['MalonylCoA']]   =  v_malCoA - v_naringen
    dy[IDX['Naringenin']]   =  v_naringen
    dy[IDX['Isoprenoid']]   =  v_isoprenoid - v_betacarot
    dy[IDX['BetaCarotene']] =  v_betacarot

    return dy


def simulate(weights, t_end=50.0, n_points=101):
    """Integrate with the sliders' weights applied."""
    p = effective_rates(weights)
    t_eval = np.linspace(0, t_end, n_points)
    sol = solve_ivp(rhs, (0, t_end), Y0, args=(p,),
                    method='LSODA', rtol=1e-6, atol=1e-9, t_eval=t_eval)
    return sol


# Quick test with all sliders at 1.0
test_weights = {m: 1.0 for m in MODULES}
sol = simulate(test_weights)
print(f"Test sim: success={sol.success}, final Current={sol.y[IDX['Current']][-1]:.3f}, "
      f"final N2={sol.y[IDX['N2']][-1]:.3f}")

Test sim: success=True, final Current=0.792, final N2=6.195


## Step 5 — Circuit selection from slider weights

In [ ]:
def score_circuits(weights):
    """Dot product of each circuit's score vector with slider weights."""
    return {name: sum(weights[m] * info['scores'][m] for m in MODULES)
            for name, info in CIRCUITS.items()}

def select_consortium(weights, threshold=0.25):
    scores = score_circuits(weights)
    selected = {n for n,s in scores.items() if s >= threshold}
    # Dependency closure
    changed = True
    while changed:
        changed = False
        for n in list(selected):
            for req in CIRCUITS[n]['requires']:
                if req not in selected:
                    selected.add(req); changed = True
    return sorted(selected, key=lambda n: -scores[n]), scores

## Step 6 — The live renderer

Draws one big plot (iBioSim-style) with a species picker, plus the
consortium table and per-circuit fitness bars. Everything updates in under
a second when a slider moves.


In [ ]:
MODULE_COLORS = {
    'cross_feeding':         '#8FD14F',
    'nitrogen_cycle':        '#A855F7',
    'electrogenesis':        '#00B4B4',
    'quorum_sensing':        '#3B82F6',
    'kill_switch':           '#F59E0B',
    'metabolite_production': '#EF4444',
}

# Preset species groupings (copy-paste ready)
SPECIES_PRESETS = {
    'iBioSim default (Electrogenesis + Kill switch)':
        ['AHL','Current','Gmet','Gsulf','KillSignal','MazE_A','OmcZ_A','OmcZ_B'],
    'Cross-feeding':
        ['Ecoli_K','Ecoli_L','Ecoli_Trp','Ecoli_His','Lys','Leu','Trp','His'],
    'Nitrogen cycle':
        ['NH4','NO2','NO3','NO','N2O','N2'],
    'Metabolite production':
        ['AcetylCoA','MalonylCoA','Naringenin','Isoprenoid','BetaCarotene'],
    'Quorum sensing':
        ['AHL','LuxR_AHL','AI2','KillSignal','MazF_A','MazF_B'],
    'All 33 species (full ecosystem)':
        list(SPECIES),
}

# A distinct color for each species, cycled
_PALETTE = [
    '#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C','#E67E22','#34495E',
    '#16A085','#D35400','#8E44AD','#27AE60','#C0392B','#2980B9','#F1C40F','#7F8C8D',
    '#E91E63','#00BCD4','#CDDC39','#795548','#607D8B','#FF5722','#009688','#FFC107',
    '#673AB7','#FF9800','#4CAF50','#F44336','#2196F3','#9C27B0','#3F51B5','#00ACC1',
    '#8BC34A',
]
SPECIES_COLOR = {s: _PALETTE[i % len(_PALETTE)] for i,s in enumerate(SPECIES)}


def render(weights, species_to_plot, threshold=0.25, t_end=50.0, log_y=False):
    """One big plot (iBioSim-style) + the consortium table + fitness bars."""
    # --- 1. Consortium table ---
    selected, scores = select_consortium(weights, threshold)
    if selected:
        md = [f"### 🧬 Selected consortium — {len(selected)} of 20 circuits\n",
              "| Score | Circuit | Module | Organism |",
              "|---|---|---|---|"]
        for n in selected:
            c = CIRCUITS[n]
            md.append(f"| **{scores[n]:.2f}** | `{n}` | `{c['module']}` | *{c['organism']}* |")
        display(Markdown('\n'.join(md)))
    else:
        display(Markdown("### ⚠️ No circuits cleared the threshold — raise some sliders."))

    # --- 2. Run simulation ---
    sol = simulate(weights, t_end=t_end)
    if not sol.success:
        print("⚠️ Integration failed:", sol.message); return

    # --- 3. ONE BIG PLOT (iBioSim TSD style) ---
    if species_to_plot:
        fig, ax = plt.subplots(figsize=(14, 7.5))
        for s in species_to_plot:
            ax.plot(sol.t, sol.y[IDX[s]],
                    label=s, color=SPECIES_COLOR[s],
                    linewidth=1.8, marker='.', markersize=3, markevery=5)
        ax.set_title(f"sim1 simulation results  (t = 0 ... {t_end:.0f})",
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('time (minutes)', fontsize=11)
        ax.set_ylabel('amount (molecules / cells)', fontsize=11)
        ax.set_xlim(0, t_end)
        ax.set_ylim(0, 5.5)
        ax.legend(loc='best', ncol=min(4, len(species_to_plot)),
                  fontsize=9, framealpha=0.9)
        ax.grid(True, alpha=0.3)
        if log_y:
            ax.set_yscale('log')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
    else:
        display(Markdown("*(no species selected for plotting)*"))

    # --- 4. Fitness bar chart ---
    fig, ax = plt.subplots(figsize=(11, 6))
    names = list(CIRCUITS.keys())
    vals  = [scores[n] for n in names]
    colors= [MODULE_COLORS[CIRCUITS[n]['module']] if n in selected else '#D0D9E0' for n in names]
    ax.barh(names, vals, color=colors, edgecolor='#0B1F2A', linewidth=0.5)
    ax.axvline(threshold, color='#EF4444', ls='--', lw=1.5, label=f'threshold = {threshold}')
    ax.invert_yaxis()
    ax.set_xlabel('Fitness score'); ax.set_xlim(0, 1.05)
    ax.legend(loc='lower right', fontsize=10)
    ax.set_title('Per-circuit fitness (coloured = selected)', fontsize=12)
    for spine in ['top','right']:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()

print("Renderer ready.")

Renderer ready.


---
## 🎛️ Tune your consortium

Each slider scales the rate constants for its module between **0 (off)** and
**1 (native strength)**. The circuit table, 2×2 simulation panel, and
fitness bars redraw after every move.

**Preset scenarios to try:**
- **Full ecosystem** → every slider at `1.0`
- **Brewery wastewater MFC** → cross-feeding `1.0`, electrogenesis `1.0`, kill switch `0.6`, everything else low
- **Municipal N-removal** → nitrogen cycle `1.0`, cross-feeding `0.3`, electrogenesis off
- **Bioproduction** → metabolite production `1.0`, cross-feeding `0.5`, quorum sensing `0.5`
- **Maximum containment** → kill switch `1.0`, quorum sensing `1.0` (QS triggers the kill switch)


In [ ]:
slider_style  = {'description_width': '200px'}
slider_layout = widgets.Layout(width='560px')

def _slider(desc, val=0.7):
    return widgets.FloatSlider(value=val, min=0, max=1, step=0.05,
                               description=desc, style=slider_style,
                               layout=slider_layout, continuous_update=False)

# --- Module sliders (the only user controls) ---
s_cross   = _slider('Cross-feeding:')
s_nitr    = _slider('Nitrogen cycle:')
s_electro = _slider('Electrogenesis:')
s_quorum  = _slider('Quorum sensing:')
s_kill    = _slider('Kill switch:')
s_prod    = _slider('Metabolite production:')

# --- Hardcoded baseline settings ---
SELECTION_CUTOFF = 0.5
SIM_TIME_HORIZON = 100.0
PLOT_SPECIES = ['AHL','Current','Gmet','Gsulf','KillSignal','MazE_A','OmcZ_A','OmcZ_B']

output_area = widgets.Output()

def on_change(change=None):
    weights = {
        'cross_feeding':         s_cross.value,
        'nitrogen_cycle':        s_nitr.value,
        'electrogenesis':        s_electro.value,
        'quorum_sensing':        s_quorum.value,
        'kill_switch':           s_kill.value,
        'metabolite_production': s_prod.value,
    }
    with output_area:
        clear_output(wait=True)
        render(weights, species_to_plot=PLOT_SPECIES,
               threshold=SELECTION_CUTOFF, t_end=SIM_TIME_HORIZON,
               log_y=False)

for w in [s_cross, s_nitr, s_electro, s_quorum, s_kill, s_prod]:
    w.observe(on_change, names='value')

controls = widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 8px 0;">Module sliders (0 = off, 1 = native rate)</h3>'),
    s_cross, s_nitr, s_electro, s_quorum, s_kill, s_prod,
])

display(controls)
display(output_area)
on_change()  # initial render

Output()